In [1]:
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding, LSTM, Dense

In [2]:
df = pd.read_csv("../dataset/processed_dataset.csv")

In [3]:
encoder = LabelEncoder()

df['class'] = encoder.fit_transform(df['class'])

In [4]:
df.head()

,text,class
0,ex wife threatening suiciderecently left wife ...,1
1,weird dont get affected compliment coming some...,0
2,finally almost never hear bad year ever swear ...,0
3,need helpjust help im cry hard,1
4,im losthello name adam ive struggling year im ...,1


In [5]:
X = df['text']

y = df['class']

In [6]:
tokenizer = Tokenizer(num_words=5000)

In [7]:
tokenizer.fit_on_texts(X)

In [8]:
X_sequences = tokenizer.texts_to_sequences(X)

In [9]:
print(X_sequences[0])

[439, 563, 2187, 140, 563, 33, 1166, 850, 1211, 26, 294, 1125, 22, 39, 20, 128, 954, 2187, 63, 428, 20, 182, 78, 4, 241, 64, 107, 39, 5, 99, 12, 3202, 819, 8, 31, 492, 18, 380, 469, 188, 350, 43, 45, 563, 273, 91, 1166, 295, 68, 1874, 1, 607, 135, 265, 20, 141, 26, 89, 303]


In [10]:
X_padded = pad_sequences(
    X_sequences,
    maxlen=100
)

In [11]:
print(X_padded.shape)

(231981, 100)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
model = Sequential()

model.add(
    Embedding(
        input_dim=5000,
        output_dim=128,
        input_length=100
    )
)
model.add(LSTM(64))

model.add(Dense(1, activation='sigmoid'))

In [17]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [18]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
2320/2320 ━━━━━━━━━━━━━━━━━━━━ 265s 111ms/step - accuracy: 0.9214 - loss: 0.2057 - val_accuracy: 0.9305 - val_loss: 0.1810
Epoch 2/5
2320/2320 ━━━━━━━━━━━━━━━━━━━━ 157s 68ms/step - accuracy: 0.9380 - loss: 0.1642 - val_accuracy: 0.9344 - val_loss: 0.1766
Epoch 3/5
2320/2320 ━━━━━━━━━━━━━━━━━━━━ 154s 66ms/step - accuracy: 0.9446 - loss: 0.1454 - val_accuracy: 0.9336 - val_loss: 0.1764
Epoch 4/5
2320/2320 ━━━━━━━━━━━━━━━━━━━━ 162s 70ms/step - accuracy: 0.9505 - loss: 0.1296 - val_accuracy: 0.9306 - val_loss: 0.1893
Epoch 5/5
2320/2320 ━━━━━━━━━━━━━━━━━━━━ 153s 66ms/step - accuracy: 0.9562 - loss: 0.1139 - val_accuracy: 0.9301 - val_loss: 0.1999


In [20]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

1450/1450 ━━━━━━━━━━━━━━━━━━━━ 23s 16ms/step - accuracy: 0.9290 - loss: 0.2044
Accuracy: 0.9289824962615967


In [21]:
y_pred = model.predict(X_test)

1450/1450 ━━━━━━━━━━━━━━━━━━━━ 23s 16ms/step


In [22]:
y_pred = (y_pred > 0.5).astype(int)

In [23]:
from sklearn.metrics import classification_report

from sklearn.metrics import confusion_matrix

In [24]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93     23145
           1       0.92      0.94      0.93     23252

    accuracy                           0.93     46397
   macro avg       0.93      0.93      0.93     46397
weighted avg       0.93      0.93      0.93     46397



In [25]:
print(confusion_matrix(y_test, y_pred))

[[21258  1887]
 [ 1408 21844]]


In [26]:
model.save("../trained_models/lstm_model.h5")

In [28]:
import pickle

with open("../trained_models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)